# Dim_dim_trade_category 

In [11]:
import pandas as pd

# =====================================================
# LOAD DATASETS
# =====================================================

# Existing trade fact table
fact_trade = pd.read_csv(
    "/Users/hamid/Desktop/SupplyChain_Project/data/curated/fact_trade_volume1.csv"
)

# Disruption era dimension
dim_era = pd.read_csv(
    "/Users/hamid/Desktop/SupplyChain_Project/data/curated/dim_disruption_era.csv"
)

# Trade category dimension
dim_trade_category = pd.read_csv(
    "/Users/hamid/Desktop/SupplyChain_Project/data/curated/dim_trade_category.csv"
)

# =====================================================
# PREPARE YEAR FOR ERA LOOKUP
# =====================================================

# Convert date_key to string
fact_trade["date_key"] = (
    fact_trade["date_key"]
    .astype(str)
)

# Extract year
fact_trade["year"] = (
    fact_trade["date_key"]
    .str[:4]
    .astype(int)
)

# =====================================================
# ERA LOOKUP FUNCTION
# =====================================================

def assign_era_sk(year, era_df):

    matched_era = era_df[
        (era_df["start_year"] <= year) &
        (era_df["end_year"] >= year)
    ]

    if not matched_era.empty:
        return matched_era.iloc[0]["era_sk"]

    return None

# =====================================================
# ADD era_sk
# =====================================================

fact_trade["era_sk"] = (
    fact_trade["year"]
    .apply(lambda x: assign_era_sk(x, dim_era))
)

# =====================================================
# CLEAN trade_category FOR JOIN
# =====================================================

fact_trade["trade_category"] = (
    fact_trade["trade_category"]
    .astype(str)
    .str.strip()
    .str.lower()
)

dim_trade_category["trade_category"] = (
    dim_trade_category["trade_category"]
    .astype(str)
    .str.strip()
    .str.lower()
)

# =====================================================
# JOIN trade_category_sk
# =====================================================

fact_trade = fact_trade.merge(
    dim_trade_category[
        ["trade_category_sk", "trade_category"]
    ],
    on="trade_category",
    how="left"
)

# =====================================================
# CREATE FACT SURROGATE KEY
# =====================================================

fact_trade.insert(
    0,
    "trade_fact_sk",
    range(1, len(fact_trade) + 1)
)

# =====================================================
# DROP UNUSED COLUMNS
# =====================================================

fact_trade.drop(
    columns=[
        "year",
        "trade_category",
        "key_goods"
    ],
    inplace=True,
    errors="ignore"
)

# =====================================================
# FINAL COLUMN ORDER
# =====================================================

fact_trade = fact_trade[
    [
        "trade_fact_sk",
        "date_key",
        "exporter_geo_sk",
        "importer_geo_sk",
        "era_sk",
        "trade_category_sk",
        "trade_value_bn_usd",
        "yoy_growth_pct",
        "effective_tariff_rate_pct",
        "trade_active",
        "supply_chain_integrated",
        "concentration_risk"
    ]
]

# =====================================================
# VALIDATION
# =====================================================

print("Missing era_sk:",
      fact_trade["era_sk"].isnull().sum())

print("Missing trade_category_sk:",
      fact_trade["trade_category_sk"].isnull().sum())

# =====================================================
# EXPORT FINAL FACT TABLE
# =====================================================

fact_trade.to_csv(
    "/Users/hamid/Desktop/SupplyChain_Project/data/curated/fact_trade_volume_enriched.csv",
    index=False
)

print("\nfact_trade_volume_enriched.csv created successfully.\n")

print(fact_trade.head())

Missing era_sk: 0
Missing trade_category_sk: 0

fact_trade_volume_enriched.csv created successfully.

   trade_fact_sk  date_key  exporter_geo_sk  importer_geo_sk  era_sk  \
0              1  20000101                1                6       1   
1              2  20000101                1                6       1   
2              3  20000101                1                6       1   
3              4  20000101                1                6       1   
4              5  20000101                1                6       1   

   trade_category_sk  trade_value_bn_usd  yoy_growth_pct  \
0                  1               23.18             0.0   
1                  3               23.18             0.0   
2                  5               23.18             0.0   
3                 17               23.18             0.0   
4                 26               23.18             0.0   

   effective_tariff_rate_pct  trade_active  supply_chain_integrated  \
0                        2.5     